# exp044 — Fine-tune de los pesos SWA públicos de nartaa (d96/384) con las etiquetas de consenso de exp031 (all-data)

Padre: exp031_raptor_etiquetas_consenso. Única variable: inicialización = pesos SWA de nartaa
(`nartaa/rsna-knee-publication-swa-weights-20261007`, `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt`) en vez de
ImageNet, con un fine-tune corto (6 épocas, lr backbone 1e-5 / cabeza 1e-4) fijado en la SPEC. Se entrena directo en
modo all-data (4349 no-gold). nartaa ya vio todos los datos: pseudo y gold solo son sanidad; decide el LB.

Notas del padre (exp031):

Padre: exp029_raptor_entreno. Única variable: etiquetas de ENTRENAMIENTO = promedio por celda de 4 fuentes blandas
(Raptor `labels_llm_soft`, L2 stevenleehans v4_blend, L3 pilkwang report_labels_v2, L4 lixin73 gpt56sol); si falta
una fuente en una celda se promedian las presentes. Todo lo demás idéntico al padre.

Notas del padre:


SPEC: `SPEC.md` en esta carpeta. Porta a celdas el script público `dreaddevelopment/knee-mri-training-the-twelve-finding-model`
(CoAtNet-2 384 + atención MIL por hallazgo, 2.5D) sobre el corpus denso 96×384 construido con `build_corpus96.py`
(script auxiliar de esta carpeta = `build_study` verbatim de `nartaa/rsna-knee-0949-anatomical-mirror`).
Cambios contra el script público (declarados en la SPEC): selección de época y SWA por **pseudo AUC L2 en val fold 0**
(nunca gold), split `split_v1_seed42`, entrada = recorte central 320 del corte de 384 (como la inferencia de nartaa).

Celdas fijas: **0** parámetros · **1** setup · **2** datos y split · **3** modelo · **4** entrenamiento · **5** evaluación ·
**6** guardar · **7** verdict.

## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp064_raptor_ft_nartaa_aug_light"
PADRE = "exp044"
FOLDS = [0]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True
ALL_DATA = True           # True: entrena con los 4349 no-gold, sin validación; SWA sobre SWA_EPOCHS fijas
SWA_EPOCHS = [3, 4, 5]         # solo ALL_DATA: épocas (0-index) a promediar, fijadas con lo aprendido en fold 0

CORPUS = "/root/corpus96_384"     # disco local del pod (build_corpus96.py)
LABELS_SOFT = "/workspace/data/labels/public/dreaddevelopment/rsna-knee-labels/labels_llm_soft.parquet"  # fuente 1 (la del padre)
L3_CSV = "/workspace/data/labels/public/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv"  # exp031: fuente 3
L4_CSV = "/workspace/data/labels/public/lixin73/rsna-knee-llm-report-labels-sol56/labels_llm_gpt56sol.csv"  # exp031: fuente 4
L2_CSV = "/workspace/data/labels/public/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"  # vara y fuente 2
V1_CSV = "/workspace/data/labels/report_labels_v1.csv"
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
INIT_CKPT = "/workspace/data/public_assets/nartaa__rsna-knee-publication-swa-weights-20261007/raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt"  # exp044: init (padre: ImageNet)
INIT_SHA256 = "7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef"  # checkpoint-manifest.json de nartaa
IMG = 384                 # lado guardado en el corpus
RES = 320                 # entrada = recorte central RES del corte IMG (como la inferencia nartaa)
NORM = "imagenet"
K = 12
K_EVAL = 24
EPOCHS = 6                # exp044: fine-tune corto (padre: 16 desde ImageNet)
BATCH_SIZE = 8
BB_LR = 1e-5              # exp044 (padre: 3e-5)
HEAD_LR = 1e-4            # exp044 (padre: 1e-3)
WEIGHT_DECAY = 0.02
PCT_START = 0.15
GRAD_CLIP = 3.0
TOPK = 3                  # SWA = media de las TOPK mejores épocas por pseudo L2
L2_THRESHOLD = 0.5
GRAD_CKPT = True
N_WORKERS = 12
LOG_EVERY = 50
AUG_STRENGTH = "light"       # exp064: aug artefactos RM por (estudio, slot): "off" (= padre exp044) | "light" | "strong"
NOTES = "Track B: exp044 (fine-tune nartaa all-data) + mri_artifact_aug light; decide el LB del blend-mine (raptor = rank mean nartaa + este)"


## 1. Setup

In [ ]:
import copy
import json
import math
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import decision, git_info, guardar_metricas, leer_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona checkpoints (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
assert not ALL_DATA or len(FOLDS) == 1
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
if SMOKE:
    EPOCHS = 1
TAG = "alldata" if ALL_DATA else "f"
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · "
      f"{torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'} · timm {timm.__version__} · torch {torch.__version__}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (las revisiones detienen el notebook si algo está mal)

In [ ]:
def _read(path, cols=None):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    return df.set_index("StudyInstanceUID")[cols or LABELS].astype(float)

split = load_split(SPLIT)
def _read_soft(path, name):
    df = pd.read_parquet(path) if str(path).endswith(".parquet") else pd.read_csv(path, low_memory=False)
    df["StudyInstanceUID"] = df.StudyInstanceUID.astype(str)
    assert not df.StudyInstanceUID.duplicated().any(), f"{name} con StudyInstanceUID duplicados"
    df = df.set_index("StudyInstanceUID")[LABELS].apply(pd.to_numeric, errors="coerce").astype(float)
    assert df.stack().between(0.0, 1.0).all(), f"{name} fuera de [0, 1]"
    return df

# exp031: etiquetas de ENTRENAMIENTO = promedio por celda de 4 fuentes blandas (si falta una fuente, media de las presentes)
SOURCES = {"raptor": _read_soft(LABELS_SOFT, "raptor"), "L2": _read_soft(L2_CSV, "L2"),
           "L3": _read_soft(L3_CSV, "L3"), "L4": _read_soft(L4_CSV, "L4")}
_all = sorted(set().union(*[s.index for s in SOURCES.values()]))
_stack = np.stack([s.reindex(_all)[LABELS].to_numpy(dtype=float) for s in SOURCES.values()])
_n_src = np.isfinite(_stack).sum(0)
soft = pd.DataFrame(np.where(_n_src > 0, np.nansum(_stack, 0) / np.maximum(_n_src, 1), np.nan),
                    index=pd.Index(_all, name="StudyInstanceUID"), columns=LABELS)
N_SRC = pd.DataFrame(_n_src, index=soft.index, columns=LABELS)
l2 = _read(L2_CSV)                    # vara interna: L2 binarizada > 0.5
v1 = _read(V1_CSV)                    # informativo
train_csv = pd.read_csv("/workspace/data/raw/train.csv")
train_csv["StudyInstanceUID"] = train_csv.StudyInstanceUID.astype(str)
gold = train_csv.dropna(subset=LABELS, how="all").set_index("StudyInstanceUID")[LABELS].astype(float)
gold_ids = list(gold.index)
assert len(gold) == 58 and set(gold_ids) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
soft = soft.dropna(how="all")         # estudio sin ninguna fuente fuera; celda sin fuentes (NaN) se enmascara en la pérdida
assert np.nanmin(soft.values) >= 0 and np.nanmax(soft.values) <= 1
print(f"etiquetas de entrenamiento (consenso): {len(soft)} estudios · celdas NaN {int(soft.isna().sum().sum())}")

corpus_ids = np.load(f"{CORPUS}/ids.npy").astype(str)
corpus_masks = np.load(f"{CORPUS}/masks.npy")
ID2ROW = {u: i for i, u in enumerate(corpus_ids)}
D = corpus_masks.shape[1]
assert D == 96, D
meta = json.load(open(f"{CORPUS}/build_meta.json"))
print(f"corpus {len(corpus_ids)} × {D} · cortes válidos medios {corpus_masks.sum(1).mean():.1f} · "
      f"estudios vacíos {(corpus_masks.sum(1) == 0).sum()} · errores build {len(meta['errors'])}")
assert set(gold_ids) <= set(ID2ROW)

pseudo_bin = (l2 > L2_THRESHOLD).astype(float).where(l2.notna())
v1_bin = v1.where(v1.isin([0.0, 1.0]))

def fold_ids(fold):
    """train / val para un fold de split_v1; train = pool sin el fold ∩ etiquetas de entrenamiento (consenso)."""
    if ALL_DATA:
        tr = [u for u in soft.index if u not in set(gold_ids) and u in ID2ROW]
        return tr, []
    pool = split[split.fold >= 0]
    tr = [u for u in pool.loc[pool.fold != fold, "StudyInstanceUID"] if u in soft.index and u in ID2ROW]
    va = [u for u in pool.loc[pool.fold == fold, "StudyInstanceUID"] if u in ID2ROW]
    return tr, va

for f in FOLDS:
    tr, va = fold_ids(f)
    assert not set(tr) & set(gold_ids), "gold dentro del entrenamiento"
    assert not set(tr) & set(va)
    print(f"fold {f}: train {len(tr)} · val {len(va)} (L2 cubre {pseudo_bin.index.isin(va).sum()}) · gold {len(gold_ids)}")
    _ns = N_SRC.reindex(tr).fillna(0).to_numpy()
    print(f"  celdas de train por nº de fuentes (0..4): {[int((_ns == k).sum()) for k in range(5)]} · "
          f"cobertura por fuente (estudios de train): {({n: int(pd.Index(tr).isin(s.index).sum()) for n, s in SOURCES.items()})}")
    _tr = {n: s.reindex(tr)[LABELS].to_numpy() for n, s in SOURCES.items()}
    _agree = lambda a, b: np.mean([(x > 0.5) == (y > 0.5) for x, y in zip(_tr[a].ravel(), _tr[b].ravel())
                                   if np.isfinite(x) and np.isfinite(y)])
    print("  acuerdo binarizado (> 0.5) en train: " + " · ".join(f"{a}–{b} {_agree(a, b):.3f}" for a, b in
          [("L2", "L4"), ("L2", "L3"), ("L3", "L4"), ("raptor", "L2"), ("raptor", "L3"), ("raptor", "L4")]))
    print(f"  media del objetivo en train: consenso {np.nanmean(soft.loc[tr].to_numpy()):.3f} · " +
          " · ".join(f"{n} {np.nanmean(v):.3f}" for n, v in _tr.items()))

## 3. Modelo (verbatim del script público: `build_backbone` + `RaptorClassifier`; dataset `StudyWindows`)

In [ ]:
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
SLOT_SIZES = [26, 22, 18, 12, 18]     # exp064 plumbing (igual que exp045): slots del corpus (Sag fluid, Sag, Cor fluid, Cor, Ax)
SLOT_OF = np.repeat(np.arange(len(SLOT_SIZES)), SLOT_SIZES)
assert len(SLOT_OF) == D


class StudyWindows(Dataset):
    """Bolsa de ventanas 2.5D por estudio (3 cortes consecutivos -> RGB) desde el corpus memmap.
    Igual que el script público salvo: recorte central RES del corte IMG (en vez de resize) y la conversión a float,
    la ganancia ×U(0.9,1.1) y la normalización ImageNet se hacen en GPU (mismas operaciones)."""
    def __init__(self, ids, k, train):
        self.ids, self.k, self.train = list(ids), k, train
        self.vols = None

    def __len__(self):
        return len(self.ids)

    def _centers(self, valid, count):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, D - 2))]
        if self.train:
            pool = cs * (count // len(cs) + 1)
            random.shuffle(pool)
            return pool[:count]
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return [cs[i] for i in idx]

    def __getitem__(self, i):
        if self.vols is None:
            self.vols = np.load(f"{CORPUS}/vols.npy", mmap_mode="r")
        uid = self.ids[i]; row = ID2ROW[uid]
        valid = np.where(corpus_masks[row] > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, D))
        cs = np.array([max(1, min(c, D - 2)) for c in self._centers(valid, self.k)])
        lo = (IMG - RES) // 2
        vol = self.vols[row]
        tri = np.stack([vol[cs - 1, lo:lo + RES, lo:lo + RES], vol[cs, lo:lo + RES, lo:lo + RES],
                        vol[cs + 1, lo:lo + RES, lo:lo + RES]], 1)          # (K,3,RES,RES) uint8
        gain = 1.0 + (random.random() - 0.5) * 0.20 if self.train else 1.0
        slot = torch.from_numpy(SLOT_OF[cs].astype(np.int64))     # exp064: slot del corte central por ventana
        return torch.from_numpy(np.ascontiguousarray(tri)), torch.tensor(gain, dtype=torch.float32), i, slot


# ---- exp064: mri_artifact_aug (verbatim de orq/p7c_augs/mri_aug.py, sin el docstring de módulo) ----
import math

import torch
import torch.nn.functional as F

N_SLOTS = 5

# p = probability per (study, slot) unit; ranges are U(lo, hi) unless noted.
# P(unit touched by >= 1 transform): light 1 - 0.9^4 * 0.95^2 = 41 %; strong 1 - 0.7*0.75*0.8*0.8*0.75*0.75 = 81 %.
PRESETS = {
    "light": {
        "bias":   {"p": 0.10, "coef": (0.05, 0.15), "grid": 4},          # field = exp(coef * f), f in [-1, 1]
        "aniso":  {"p": 0.10, "factor": (1.25, 2.0)},                     # down/up-sample one in-plane axis
        "motion": {"p": 0.05, "events": (1, 2), "shift_px": 1.5},         # per event dy, dx ~ U(-s, s) px
        "ghost":  {"p": 0.05, "n_ghosts": (4, 10), "intensity": (0.10, 0.30), "restore": 0.02},
        "rician": {"p": 0.10, "sigma": (0.005, 0.02)},
        "gamma":  {"p": 0.10, "log_gamma": 0.15},                         # gamma = exp(U(-a, a)) -> [0.86, 1.16]
    },
    "strong": {
        "bias":   {"p": 0.30, "coef": (0.10, 0.30), "grid": 4},
        "aniso":  {"p": 0.25, "factor": (1.5, 3.0)},
        "motion": {"p": 0.20, "events": (1, 3), "shift_px": 3.0},
        "ghost":  {"p": 0.20, "n_ghosts": (4, 10), "intensity": (0.20, 0.50), "restore": 0.02},
        "rician": {"p": 0.25, "sigma": (0.01, 0.05)},
        "gamma":  {"p": 0.25, "log_gamma": 0.35},                         # [0.70, 1.42]
    },
}
TRANSFORMS = ("bias", "aniso", "motion", "ghost", "rician", "gamma")
MAX_EVENTS = 3
BIAS_GRID = 4


def _resolve(strength):
    if strength is None or strength is False or strength == "off":
        return None
    cfg = PRESETS[strength] if isinstance(strength, str) else strength
    if all(cfg.get(t, {}).get("p", 0.0) <= 0 for t in TRANSFORMS):
        return None
    return cfg


def _u(g, n, lo, hi):
    return lo + (hi - lo) * torch.rand(n, generator=g)


def _draw(cfg, U, g):
    """All per-unit parameters, drawn on CPU in a fixed order (count independent of cfg values)."""
    c = {t: cfg.get(t, {"p": 0.0}) for t in TRANSFORMS}
    P = {}
    for t in TRANSFORMS:
        P[t + "_on"] = torch.rand(U, generator=g) < c[t]["p"]
    b = c["bias"]
    P["bias_coef"] = _u(g, U, *b.get("coef", (0.0, 0.0)))
    P["bias_coarse"] = torch.randn(U, 1, BIAS_GRID, BIAS_GRID, generator=g)
    a = c["aniso"]
    P["aniso_factor"] = _u(g, U, *a.get("factor", (1.0, 1.0)))
    P["aniso_axis_h"] = torch.rand(U, generator=g) < 0.5
    P["pe_axis_h"] = torch.rand(U, generator=g) < 0.5                     # phase-encoding axis for motion + ghost
    m = c["motion"]
    lo, hi = m.get("events", (1, 1))
    n_ev = lo + (torch.rand(U, generator=g) * (hi - lo + 1)).long().clamp_max(hi - lo)
    P["motion_t"] = torch.rand(U, MAX_EVENTS, generator=g)                # event times in acquisition order (0..1)
    d = (torch.rand(U, MAX_EVENTS, 2, generator=g) * 2 - 1) * m.get("shift_px", 0.0)
    P["motion_d"] = d * (torch.arange(MAX_EVENTS).view(1, -1, 1) < n_ev.view(-1, 1, 1))
    gh = c["ghost"]
    lo, hi = gh.get("n_ghosts", (4, 4))
    P["ghost_n"] = (lo + (torch.rand(U, generator=g) * (hi - lo + 1)).long().clamp_max(hi - lo)).clamp_min(2)
    P["ghost_int"] = _u(g, U, *gh.get("intensity", (0.0, 0.0)))
    P["ghost_restore"] = float(gh.get("restore", 0.02))
    P["rician_sigma"] = _u(g, U, *c["rician"].get("sigma", (0.0, 0.0)))
    a = c["gamma"].get("log_gamma", 0.0)
    P["gamma"] = torch.exp(_u(g, U, -a, a))
    P["device_seed"] = int(torch.randint(0, 2 ** 31 - 1, (1,), generator=g))
    return P


def _kspace_lines(P, units, H, W, dev):
    """Complex multiplier (n, H, W) for k-space with phase-encoding along H (rows): ghosting x motion phase ramps."""
    fy = torch.fft.fftfreq(H, device=dev)                                 # cycles / pixel, unshifted order
    fx = torch.fft.fftfreq(W, device=dev)
    line = torch.round(fy * H).long() + H // 2                           # 0..H-1 in acquisition (centred) order
    n = len(units)
    mult = torch.ones(n, H, W, dtype=torch.complex64, device=dev)
    gon = P["ghost_on"][units].to(dev)
    if gon.any():
        ng = P["ghost_n"][units].to(dev).view(n, 1)
        gi = P["ghost_int"][units].to(dev).view(n, 1)
        keep = (fy.abs() <= P["ghost_restore"]).view(1, H)
        mask = torch.where((line.view(1, H) % ng == 0) & ~keep, 1.0 - gi, torch.ones_like(gi))
        mask = torch.where(gon.view(n, 1), mask, torch.ones_like(mask))
        mult = mult * mask.view(n, H, 1)
    mon = P["motion_on"][units].to(dev)
    if mon.any():
        tau = (line.float() + 0.5) / H                                    # acquisition time of each PE line
        t = P["motion_t"][units].to(dev)                                  # (n, E)
        d = P["motion_d"][units].to(dev) * mon.view(n, 1, 1)              # (n, E, 2) px
        step = (tau.view(1, H, 1) >= t.view(n, 1, -1)).float()            # (n, H, E)
        disp = torch.einsum("nhe,nec->nhc", step, d)                      # (n, H, 2) cumulative (dy, dx) per line
        disp = disp - disp[:, H // 2:H // 2 + 1]                          # de-centre (Shaw 2019 "de-median"): the
        # k-space centre line keeps position 0, so motion gives ghosts/blur instead of a global (wrapping) shift
        ph = fy.view(1, H, 1) * disp[..., 0:1] + fx.view(1, 1, W) * disp[..., 1:2]
        mult = mult * torch.polar(torch.ones_like(ph), -2 * math.pi * ph)
    return mult


@torch.no_grad()
def mri_artifact_aug(x, slot_ids, strength="light", generator=None, n_slots=N_SLOTS, return_params=False):
    cfg = _resolve(strength)
    if cfg is None:
        return (x, None) if return_params else x
    B, K, C, H, W = x.shape
    dev, dtype = x.device, x.dtype
    U = B * n_slots
    P = _draw(cfg, U, generator)
    unit = (torch.arange(B).view(B, 1) * n_slots + slot_ids.detach().long().cpu().clamp(0, n_slots - 1)).view(-1)
    unit_d = unit.to(dev)
    gd = torch.Generator(device=dev).manual_seed(P["device_seed"])
    y = x.reshape(B * K, C, H, W).float().clone()

    def windows(on):                                                       # window indices whose unit is active
        return torch.nonzero(on[unit]).flatten().to(dev)

    # 1. multiplicative bias field, one smooth field per (study, slot)
    w = windows(P["bias_on"])
    if len(w):
        uu = unit_d[w]
        f = F.interpolate(P["bias_coarse"].to(dev)[uu], size=(H, W), mode="bicubic", align_corners=True)
        f = f / f.abs().amax(dim=(2, 3), keepdim=True).clamp_min(1e-6)
        y[w] = y[w] * torch.exp(P["bias_coef"].to(dev)[uu].view(-1, 1, 1, 1) * f)
    # 2. reduced in-plane resolution: area down-sampling of one axis by factor f, bilinear back
    for u in torch.nonzero(P["aniso_on"]).flatten().tolist():
        w = torch.nonzero(unit == u).flatten().to(dev)
        if not len(w):
            continue
        fct = float(P["aniso_factor"][u])
        size = (max(8, int(round(H / fct))), W) if bool(P["aniso_axis_h"][u]) else (H, max(8, int(round(W / fct))))
        lo = F.interpolate(y[w], size=size, mode="area")
        y[w] = F.interpolate(lo, size=(H, W), mode="bilinear", align_corners=False)
    # 3. k-space: ghosting (periodic PE-line attenuation) + motion (phase ramps on the lines acquired after each event)
    w = windows(P["ghost_on"] | P["motion_on"])
    if len(w):
        uu = unit[w.cpu()]
        z = y[w]
        tr = ~P["pe_axis_h"][uu].to(dev)                                   # PE along W -> transpose so it is rows
        z = torch.where(tr.view(-1, 1, 1, 1), z.transpose(-1, -2), z) if H == W else z
        k = torch.fft.fft2(z) * _kspace_lines(P, uu, H, W, dev).unsqueeze(1)
        z = torch.fft.ifft2(k).abs()
        z = torch.where(tr.view(-1, 1, 1, 1), z.transpose(-1, -2), z) if H == W else z
        y[w] = z
    y.clamp_(0, 1)
    # 4. Rician noise (magnitude of complex Gaussian), sigma per unit
    w = windows(P["rician_on"])
    if len(w):
        s = P["rician_sigma"].to(dev)[unit_d[w]].view(-1, 1, 1, 1)
        z = y[w]
        n1 = torch.randn(z.shape, generator=gd, device=dev)
        n2 = torch.randn(z.shape, generator=gd, device=dev)
        y[w] = torch.sqrt((z + s * n1) ** 2 + (s * n2) ** 2)
        y.clamp_(0, 1)
    # 5. gamma per unit
    w = windows(P["gamma_on"])
    if len(w):
        y[w] = y[w].pow(P["gamma"].to(dev)[unit_d[w]].view(-1, 1, 1, 1))
    out = y.clamp_(0, 1).view(B, K, C, H, W).to(dtype)
    return (out, P) if return_params else out
# ---- fin mri_aug.py ----

_AUG_T = []   # exp064: ms por paso de la aug (primeros 20 pasos, con cuda.synchronize)


def to_input(x_u8, gain, slot=None, train=False):
    x = x_u8.to(DEVICE, non_blocking=True).float() / 255.0
    if train and AUG_STRENGTH != "off":     # exp064: artefactos RM solo en train, antes de la ganancia y la normalización
        timing = len(_AUG_T) < 20 and DEVICE.type == "cuda"
        if timing:
            torch.cuda.synchronize(); _t = time.time()
        x = mri_artifact_aug(x, slot.to(DEVICE, non_blocking=True), AUG_STRENGTH)
        if timing:
            torch.cuda.synchronize(); _AUG_T.append((time.time() - _t) * 1000)
            if len(_AUG_T) == 20:
                print(f"[aug] mri_artifact_aug {AUG_STRENGTH}: {np.mean(_AUG_T[2:]):.1f} ms/paso (media pasos 3-20)", flush=True)
    if not torch.all(gain == 1.0):
        x = (x * gain.to(DEVICE).view(-1, 1, 1, 1, 1)).clamp(0, 1)
    if NORM == "imagenet":
        x = (x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)
    return x


def build_backbone(arch, pretrained=True):
    return timm.create_model(arch, pretrained=pretrained, num_classes=0, in_chans=3, global_pool="avg", img_size=RES)


class RaptorClassifier(nn.Module):
    """Raptor encoder + per-diagnosis attention-MIL head (12 findings)."""
    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K_ = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).view(B, K_, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.encode(x))


_INIT = None
def _init_state():
    """exp044: pesos SWA de nartaa. Misma arquitectura (CoAtNet-2 rmlp img_size 320 + RaptorClassifier), carga estricta."""
    global _INIT
    if _INIT is None:
        import hashlib
        with open(INIT_CKPT, "rb") as fh:
            sha = hashlib.file_digest(fh, "sha256").hexdigest()
        assert sha == INIT_SHA256, f"sha256 del checkpoint de nartaa no coincide: {sha}"
        ck = torch.load(INIT_CKPT, map_location="cpu", weights_only=False)
        assert ck.get("arch", ARCH) == ARCH, ck.get("arch")
        assert list(ck.get("lab", LABELS)) == list(LABELS), f"orden de hallazgos distinto: {ck.get('lab')}"
        print(f"[init] {Path(INIT_CKPT).name} · sha256 ok · arch {ck.get('arch')} · res {ck.get('res')} · "
              f"swa_over {ck.get('swa_over', ck.get('epoch'))} · gold (metadatos, seleccionado en gold) {ck.get('gold_auc')}", flush=True)
        _INIT = ck["model"]
    return _INIT


def build_model():
    bb = build_backbone(ARCH, pretrained=not INIT_CKPT)
    if GRAD_CKPT:
        bb.set_grad_checkpointing(True)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    if INIT_CKPT:
        missing, unexpected = model.load_state_dict(_init_state(), strict=True), None
    return model.to(DEVICE)

## 4. Entrenamiento (bucle del script público; selección y top-K por pseudo L2 en val, nunca gold)

In [ ]:
def loader(ids, k, train, bs):
    return DataLoader(StudyWindows(ids, k, train), batch_size=bs, shuffle=train, drop_last=train,
                      num_workers=N_WORKERS, pin_memory=True, persistent_workers=False)


@torch.no_grad()
def predict(model, ids):
    model.eval()
    if not ids:
        return pd.DataFrame(columns=LABELS)
    out = np.zeros((len(ids), len(LABELS)), np.float32)
    for x, g, idx, _slot in loader(ids, K_EVAL, False, max(2, BATCH_SIZE // 2)):     # exp064: eval sin aug
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            out[idx.numpy()] = torch.sigmoid(model(to_input(x, g)).float()).cpu().numpy()
    return pd.DataFrame(out, index=pd.Index(ids, name="StudyInstanceUID"), columns=LABELS)


def score(pred):
    r = {}
    for name, ref in (("pseudo", pseudo_bin), ("v1", v1_bin), ("gold", gold)):
        ids = [u for u in pred.index if u in ref.index]
        r[name] = mean_auc(ref.loc[ids, LABELS].to_numpy(), pred.loc[ids, LABELS].to_numpy()) if ids else None
    return r


def average_states(states):
    avg = {}
    for k_ in states[0]:
        v0 = states[0][k_]
        avg[k_] = (sum(s[k_].double() for s in states) / len(states)).to(v0.dtype) if v0.is_floating_point() else v0.clone()
    return avg


def save_preds(pred_val, pred_gold, name):
    if len(pred_val):
        pred_val.add_prefix("pred_").reset_index().to_csv(RESULTS / f"oof_{name}.csv", index=False)
    pred_gold.add_prefix("pred_").reset_index().to_csv(RESULTS / f"gold_{name}.csv", index=False)


def train_fold(fold):
    seed_everything(SEED + fold)
    tr_ids, va_ids = fold_ids(fold)
    if SMOKE:
        tr_ids, va_ids = tr_ids[:64], va_ids[:48]
    sfx = f"{TAG}{'' if ALL_DATA else fold}"
    tl = loader(tr_ids, K, True, BATCH_SIZE)
    model = build_model()
    prev = np.clip(np.nanmean(soft.loc[tr_ids, LABELS].to_numpy(), 0), 0.03, 0.7)
    lossf = nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(np.clip((1 - prev) / prev, 1, 10), dtype=torch.float32, device=DEVICE))
    Y = {u: torch.tensor(soft.loc[u, LABELS].to_numpy(), dtype=torch.float32) for u in tr_ids}
    head_p = [p for n_, p in model.named_parameters() if not n_.startswith("backbone.")]
    bb_p = [p for n_, p in model.named_parameters() if n_.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": bb_p, "lr": BB_LR}, {"params": head_p, "lr": HEAD_LR}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[BB_LR, HEAD_LR], total_steps=max(1, len(tl) * EPOCHS),
                                                pct_start=PCT_START)
    print(f"[{sfx}] train {len(tr_ids)} · val {len(va_ids)} · {len(tl)} pasos/época · pos_weight "
          f"{np.round(np.clip((1 - prev) / prev, 1, 10), 2).tolist()}", flush=True)
    history, topk, best, keep = [], [], None, {}
    start = time.time()
    for epoch in range(EPOCHS):
        model.train(); tot = 0.0; t_ep = time.time()
        for step, (x, g, idx, s_) in enumerate(tl):
            y = torch.stack([Y[tl.dataset.ids[i]] for i in idx.tolist()]).to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                logits = model(to_input(x, g, s_, train=True))
            m = ~torch.isnan(y)                       # NaN enmascarado (= media BCE del padre si no hay NaN)
            loss = (lossf(logits.float(), torch.nan_to_num(y)) * m).sum() / m.sum().clamp(min=1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step(); sched.step(); tot += loss.item()
            if (step + 1) % LOG_EVERY == 0:
                el = time.time() - t_ep
                print(f"  ep{epoch} step {step + 1}/{len(tl)} loss {tot / (step + 1):.4f} · {el / (step + 1):.2f} s/paso · "
                      f"ETA época {(len(tl) - step - 1) * el / (step + 1) / 60:.1f} min", flush=True)
        t_train = time.time() - t_ep
        pv, pg = predict(model, va_ids), predict(model, gold_ids)
        sv, sg = score(pv), score(pg)
        rec = {"epoch": epoch, "train_loss": tot / max(1, len(tl)),
               "pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None,
               "v1_auc": sv["v1"]["mean"] if sv["v1"] else None,
               "gold_auc": sg["gold"]["mean"], "train_s": t_train, "eval_s": time.time() - t_ep - t_train,
               "minutes": (time.time() - start) / 60}
        history.append(rec)
        pd.DataFrame(history).to_csv(RESULTS / f"history_{sfx}.csv", index=False)
        print(f"[{sfx}] ep{epoch} loss {rec['train_loss']:.4f} | pseudo L2 {rec['pseudo_auc'] or float('nan'):.4f} | "
              f"v1 {rec['v1_auc'] or float('nan'):.4f} | gold {rec['gold_auc']:.4f} (solo veto) | train {t_train:.0f}s "
              f"eval {rec['eval_s']:.0f}s | {rec['minutes']:.1f} min", flush=True)
        state = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}
        if ALL_DATA:
            if epoch in SWA_EPOCHS:
                keep[epoch] = state
            torch.save({"model": state, "arch": ARCH, "res": RES, "lab": LABELS, "epoch": epoch, "exp": EXP},
                       RESULTS / f"last_{sfx}.pt")
            continue
        crit = rec["pseudo_auc"]
        if best is None or crit > best["pseudo_auc"]:
            best = dict(rec)
            torch.save({"model": state, "arch": ARCH, "res": RES, "lab": LABELS, "epoch": epoch, "exp": EXP,
                        "pseudo_auc": crit, "params": PARAMS}, RESULTS / f"best_{sfx}.pt")
            save_preds(pv, pg, f"best_{sfx}")
            best["pseudo_per_label"], best["gold_per_label"] = sv["pseudo"]["per_label"], sg["gold"]["per_label"]
        topk.append((crit, epoch, state)); topk.sort(key=lambda t: -t[0]); del topk[TOPK:]
        del state
    gpu_h = (time.time() - start) / 3600
    if ALL_DATA:
        eps = sorted(keep) or [EPOCHS - 1]
        states = [keep[e] for e in sorted(keep)] or [model.state_dict()]
    else:
        eps = [e for _, e, _ in topk]
        states = [s for _, _, s in topk]
    avg = average_states(states)
    model.load_state_dict(avg)
    pv, pg = predict(model, va_ids), predict(model, gold_ids)
    sv, sg = score(pv), score(pg)
    torch.save({"model": avg, "arch": ARCH, "res": RES, "lab": LABELS, "swa_over": eps, "exp": EXP, "params": PARAMS},
               RESULTS / f"swa_{sfx}.pt")
    save_preds(pv, pg, f"swa_{sfx}")
    swa = {"epochs": eps, "pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None,
           "v1_auc": sv["v1"]["mean"] if sv["v1"] else None, "gold_auc": sg["gold"]["mean"],
           "pseudo_per_label": sv["pseudo"]["per_label"] if sv["pseudo"] else None, "gold_per_label": sg["gold"]["per_label"]}
    print(f"[{sfx}] SWA sobre épocas {eps} | pseudo L2 {swa['pseudo_auc'] or float('nan'):.4f} | "
          f"v1 {swa['v1_auc'] or float('nan'):.4f} | gold {swa['gold_auc']:.4f}", flush=True)
    out = {"n_train": len(tr_ids), "n_val": len(va_ids), "history": history, "swa": swa, "gpu_hours": gpu_h,
           "swa_ckpt_path": str(RESULTS / f"swa_{sfx}.pt")}
    if best is not None:
        out.update({"pseudo_auc": best["pseudo_auc"], "v1_auc": best["v1_auc"], "gold_auc": best["gold_auc"],
                    "best_epoch": best["epoch"], "pseudo_per_label": best["pseudo_per_label"],
                    "gold_per_label": best["gold_per_label"], "ckpt_path": str(RESULTS / f"best_{sfx}.pt")})
    return out


# exp044: sanidad de la carga — el modelo inicial (nartaa) sobre gold, k_eval de la receta (gold contaminado: nartaa seleccionó en gold)
if INIT_CKPT:
    _m = build_model()
    _g = score(predict(_m, gold_ids))["gold"]["mean"]
    print(f"[init] gold-58 del modelo inicial (k_eval {K_EVAL}, sin TTA): {_g:.4f}", flush=True)
    INIT_GOLD = _g
    del _m; torch.cuda.empty_cache()

PARAMS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}
per_fold = {}
for fold in FOLDS:
    per_fold["alldata" if ALL_DATA else str(fold)] = train_fold(fold)

## 5. Evaluación (pseudo L2 decide; v1 y gold informativos/veto)

In [ ]:
rows = {}
for f, r in per_fold.items():
    rows[f"{f} mejor época ({r.get('best_epoch')})"] = {"pseudo L2": r.get("pseudo_auc"), "v1": r.get("v1_auc"), "gold (veto)": r.get("gold_auc")}
    rows[f"{f} SWA {r['swa']['epochs']}"] = {"pseudo L2": r["swa"]["pseudo_auc"], "v1": r["swa"]["v1_auc"], "gold (veto)": r["swa"]["gold_auc"]}
summary = pd.DataFrame(rows).T
display(summary.round(4))
parent = (leer_metricas(PADRE) if PADRE else None) or {}
r0 = per_fold.get("0", {})
parent_f0 = parent.get("per_fold", {}).get("0", {}).get("pseudo_auc")
delta_pseudo = r0.get("pseudo_auc") - parent_f0 if parent_f0 is not None and r0.get("pseudo_auc") is not None else None
parent_gold = parent.get("per_fold", {}).get("0", {}).get("gold_auc")
delta_gold = r0.get("gold_auc") - parent_gold if parent_gold is not None and r0.get("gold_auc") is not None else None
DECISION = decision(delta_pseudo, delta_gold)
print("Decisión automática:", DECISION, "(sin padre: es una reproducción; ver criterios absolutos en SPEC)")
for f, r in per_fold.items():
    display(pd.DataFrame(r["history"]).set_index("epoch").round(4))

## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else ("alldata" if ALL_DATA else "screen"),
    "select_on": "pseudo_L2_bin>0.5", "params": PARAMS, "per_fold": per_fold,
    "gold_auc": r0.get("gold_auc"), "gold_per_label": r0.get("gold_per_label"), "oof_pseudo_auc": None,
    "swa_f0": r0.get("swa"),
    "delta": {"pseudo_f0": delta_pseudo, "gold": delta_gold},
    "decision_auto": DECISION,
    "aug_strength": AUG_STRENGTH, "mri_aug_preset": PRESETS.get(AUG_STRENGTH),
    "mri_aug_ms_per_step": float(np.mean(_AUG_T[2:])) if len(_AUG_T) > 2 else None,
    "init_ckpt": INIT_CKPT, "init_gold_auc": globals().get("INIT_GOLD"),
    "corpus": {"path": CORPUS, "n": len(corpus_ids), "D": D, "img": IMG, "build_seconds": meta.get("seconds"),
               "n_errors": len(meta["errors"])},
    "gpu_hours_total": sum(f["gpu_hours"] for f in per_fold.values()),
}
# all-data no pisa el metrics.json del fold 0: va a results/<EXP>/alldata/metrics.json
print("Guardado:", guardar_metricas(RESULTS / "alldata" if ALL_DATA else RESULTS, metrics))

## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza la fila en `docs/EXPERIMENTOS.md`
(estado `worked`, `didnt` o `killed`). Después: commit y PR a `main`.